# Lab 06: Residual Learning and Transfer Learning

            **Duration:** 3 hours  
            **Lecture alignment:** Week 6 — Residual networks and transfer learning  
            **CLO mapping:** CLO-1, CLO-2, CLO-3, CLO-4  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Implement and shape-test a residual block.
- Pretrain on a self-generated source task and transfer representations to a small target task.
- Compare frozen, partially fine-tuned, and scratch strategies.

            ## Three-hour activity plan

            - 0–25 min: source/target dataset audit
- 25–65 min: residual block and gradient pathway
- 65–110 min: source pretraining
- 110–155 min: frozen transfer and partial fine-tuning
- 155–180 min: scratch comparison, checks, and recommendation


## Book grounding

            - Zhang, Lipton, Li, and Smola, *Dive into Deep Learning*, Cambridge University Press, 2024.
- Prince, *Understanding Deep Learning*, MIT Press, 2023.
- Bishop and Bishop, *Deep Learning: Foundations and Concepts*, Springer, 2024.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20266
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_06")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_06"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 6, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Will transferred features help more than scratch training when only 80 target examples are available? State what outcome would falsify your prediction.

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Synthetic source and target domains


In [ ]:
def make_patterns(n, task="source", size=16):
    classes = 4 if task == "source" else 2
    labels = torch.randint(0, classes, (n,)); images = .06*torch.randn(n, 1, size, size)
    for i, label in enumerate(labels.tolist()):
        if task == "source":
            if label == 0: images[i,0,2:-2,7:9] += 1
            elif label == 1: images[i,0,7:9,2:-2] += 1
            elif label == 2:
                for j in range(2,14): images[i,0,j,j] += 1
            else:
                for j in range(2,14): images[i,0,j,15-j] += 1
        elif label == 0:  # plus: composition of source features
            images[i,0,2:-2,7:9] += 1; images[i,0,7:9,2:-2] += 1
        else:  # X: composition of diagonal source features
            for j in range(2,14): images[i,0,j,j] += 1; images[i,0,j,15-j] += 1
        images[i] = torch.roll(images[i], shifts=(int(torch.randint(-1,2,(1,))), int(torch.randint(-1,2,(1,)))), dims=(1,2))
    return images.clamp(0,1), labels

source_X, source_y = make_patterns(800 if FAST_MODE else 3200, "source")
target_train_X, target_train_y = make_patterns(80 if FAST_MODE else 500, "target")
target_test_X, target_test_y = make_patterns(240 if FAST_MODE else 900, "target")


## Activity 2 — Residual block and source pretraining


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels=12):
        super().__init__(); self.c1 = nn.Conv2d(channels, channels, 3, padding=1); self.c2 = nn.Conv2d(channels, channels, 3, padding=1)
    def forward(self, x): return F.relu(x + self.c2(F.relu(self.c1(x))))

class ResidualClassifier(nn.Module):
    def __init__(self, classes):
        super().__init__(); self.stem = nn.Conv2d(1, 12, 3, padding=1); self.block = ResidualBlock(12); self.head = nn.Linear(12, classes)
    def encode(self, x):
        x = F.max_pool2d(F.relu(self.stem(x)), 2); x = self.block(x)
        return F.adaptive_avg_pool2d(x, 1).flatten(1)
    def forward(self, x): return self.head(self.encode(x))

def train_tensor_model(model, X, y, epochs, lr=.015, trainable=None):
    model = model.to(DEVICE)
    parameters = list(trainable) if trainable is not None else list(model.parameters())
    opt = torch.optim.Adam(parameters, lr=lr)
    loader = DataLoader(TensorDataset(X,y), batch_size=32, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    history=[]
    for _ in range(epochs):
        model.train(); total=0
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad(); loss=F.cross_entropy(model(xb),yb); loss.backward(); opt.step(); total+=loss.item()*len(xb)
        history.append(total/len(X))
    return history

source_model = ResidualClassifier(4)
source_history = train_tensor_model(source_model, source_X, source_y, 5 if FAST_MODE else 16)
source_model.eval()
with torch.no_grad(): source_acc=(source_model(source_X.to(DEVICE)).argmax(1).cpu()==source_y).float().mean().item()
print({"source_accuracy": source_acc, "source_final_loss": source_history[-1]})


## Activity 3 — Frozen transfer, partial fine-tuning, and scratch baseline


In [ ]:
import copy
transfer = copy.deepcopy(source_model); transfer.head = nn.Linear(12,2).to(DEVICE)
for p in transfer.parameters(): p.requires_grad=False
for p in transfer.head.parameters(): p.requires_grad=True
frozen_history = train_tensor_model(transfer, target_train_X, target_train_y, 10 if FAST_MODE else 30, trainable=transfer.head.parameters())

for p in transfer.block.parameters(): p.requires_grad=True
finetune_history = train_tensor_model(transfer, target_train_X, target_train_y, 4 if FAST_MODE else 12, lr=.004,
                                       trainable=[p for p in transfer.parameters() if p.requires_grad])
scratch = ResidualClassifier(2)
scratch_history = train_tensor_model(scratch, target_train_X, target_train_y, 14 if FAST_MODE else 42)

def accuracy(model):
    model.eval()
    with torch.no_grad(): return (model(target_test_X.to(DEVICE)).argmax(1).cpu()==target_test_y).float().mean().item()
transfer_acc, scratch_acc = accuracy(transfer), accuracy(scratch)
comparison={"transfer_accuracy":transfer_acc,"scratch_accuracy":scratch_acc,
            "frozen_trainable":sum(p.numel() for p in transfer.head.parameters()),
            "all_parameters":sum(p.numel() for p in transfer.parameters())}
print(json.dumps(comparison,indent=2))
torch.save(transfer.state_dict(), ARTIFACT_DIR/"fine_tuned_resnet.pt")
fig,ax=plt.subplots(figsize=(7,3.5)); ax.plot(frozen_history,label="frozen head"); ax.plot(range(len(frozen_history),len(frozen_history)+len(finetune_history)),finetune_history,label="partial fine-tune")
ax.plot(scratch_history,label="scratch"); ax.legend(); ax.set(title="Target-task training",xlabel="epoch",ylabel="loss"); fig.tight_layout()
fig.savefig(ARTIFACT_DIR/"transfer_comparison.png",dpi=150); plt.show()


## Automated checks


In [ ]:
probe = torch.randn(3,12,8,8); assert ResidualBlock()(probe).shape == probe.shape
assert source_acc > .70 and max(transfer_acc,scratch_acc) > .75
assert sum(p.requires_grad for p in transfer.stem.parameters()) == 0
assert (ARTIFACT_DIR/"fine_tuned_resnet.pt").exists()
print("All Lab 06 checks passed.")


## Deliverables

                - Residual-block unit test
- Source checkpoint and fine-tuned target checkpoint
- Transfer-versus-scratch plot with accuracy/parameter comparison

                Submit the executed notebook and the files created in `/content/artifacts/lab_06/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    noisy = (target_test_X + .25*torch.randn_like(target_test_X)).clamp(0,1)
    with torch.no_grad(): noisy_acc=(transfer(noisy.to(DEVICE)).argmax(1).cpu()==target_test_y).float().mean().item()
    print({"clean":transfer_acc,"noisy":noisy_acc})
else:
    print("Extension disabled: evaluate domain shift and use discriminative learning rates.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
